# FaceNet (trained on VGGFace2)

In [3]:
!pip3 install facenet-pytorch mtcnn torch torchvision

Defaulting to user installation because normal site-packages is not writeable
  Using cached facenet_pytorch-2.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached torchvision-0.24.1-cp313-cp313-win_amd64.whl.metadata (5.9 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + C:\Python313\python.exe C:\Users\user\AppData\Local\Temp\pip-install-6g8voofg\numpy_69583fd110454ad9ad3c7e1efcc5f451\vendored-meson\meson\meson.py setup C:\Users\user\AppData\Local\Temp\pip-install-6g8voofg\numpy_69583fd110454ad9ad3c7e1efcc5f451 C:\Users\user\AppData\Local\Temp\pip-install-6g8voofg\numpy_69583fd110454ad9ad3c7e1efcc5f451\.mesonpy-gxjjow56 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\user\AppData\Local\Temp\pip-install-6g8voofg\numpy_69583fd110454ad9ad3c7e1efcc5f451\.mesonpy-gxjjow56\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\user\AppData\Local\Temp\pip-install-6g8voofg\numpy_69583fd110454ad9ad3c7e1efcc5f451
      Build dir: C:\Users\user\AppData\Local\Temp\pip-install-6g8voofg\numpy_69583fd110454ad9ad3c7e1efcc5f451\.me

In [2]:
import os
import cv2
import torch
import numpy as np
import joblib
from collections import Counter
from facenet_pytorch import InceptionResnetV1, MTCNN

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Face detector
mtcnn = MTCNN(image_size=160, margin=20, device=device)

# FaceNet (pretrained on VGGFace2)
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

FACEBANK_PATH = "facebank_facenet.pkl"


ModuleNotFoundError: No module named 'facenet_pytorch'

In [5]:
def get_embedding(img):
    face = mtcnn(img)

    if face is None:
        return None

    face = face.unsqueeze(0).to(device)
    emb = model(face).detach().cpu().numpy()[0]

    # Normalize
    emb = emb / np.linalg.norm(emb)
    return emb

In [6]:
def save_facebank(names, embeddings, path=FACEBANK_PATH):
    joblib.dump({"names": names, "embeddings": embeddings}, path)
    print(f"[INFO] Saved facebank → {path}")


def load_facebank(path=FACEBANK_PATH):
    data = joblib.load(path)
    return data["names"], data["embeddings"]


In [7]:
def build_facebank(facebank_dir="facebank"):
    names = []
    embeddings = []

    for person_name in os.listdir(facebank_dir):
        person_dir = os.path.join(facebank_dir, person_name)
        if not os.path.isdir(person_dir):
            continue

        person_embs = []

        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue

            emb = get_embedding(img)
            if emb is not None:
                person_embs.append(emb)

        if len(person_embs) == 0:
            print(f"[WARN] No face for: {person_name}")
            continue

        # Average embedding per identity
        mean_emb = np.mean(person_embs, axis=0)
        mean_emb = mean_emb / np.linalg.norm(mean_emb)

        embeddings.append(mean_emb)
        names.append(person_name)

        print(f"Added identity: {person_name}")

    embeddings = np.stack(embeddings)

    save_facebank(names, embeddings)
    return names, embeddings


In [8]:
def recognize_face_knn(img_path, k=3, threshold=0.5):
    facebank_names, facebank_embeddings = load_facebank()

    img = cv2.imread(img_path)
    if img is None:
        return "Image read error", 0.0

    test_emb = get_embedding(img)
    if test_emb is None:
        return "No face detected", 0.0

    # Normalize stored embeddings
    facebank_embeddings = facebank_embeddings / np.linalg.norm(
        facebank_embeddings, axis=1, keepdims=True
    )

    # Cosine similarities
    sims = np.dot(facebank_embeddings, test_emb)

    # Top-K nearest neighbors
    topk_idxs = np.argsort(sims)[-k:][::-1]
    topk_labels = [facebank_names[i] for i in topk_idxs]
    topk_scores = [sims[i] for i in topk_idxs]

    print(f"\nTop-{k} neighbors:")
    for idx in topk_idxs:
        print(f"  {facebank_names[idx]}: {sims[idx]:.4f}")

    # Majority voting
    best_label = Counter(topk_labels).most_common(1)[0][0]
    confidence = np.mean([
        score for score, label in zip(topk_scores, topk_labels)
        if label == best_label
    ])

    print(f"> Prediction: {best_label}, confidence={confidence:.4f}\n")

    if confidence < threshold:
        return "Unknown", confidence

    return best_label, confidence


In [ ]:
build_facebank("facebank")


Added identity: Akshay Kumar
Added identity: Alexandra Daddario
Added identity: Alia Bhatt
Added identity: Amitabh Bachchan
Added identity: Andy Samberg
Added identity: Anushka Sharma


In [ ]:
name, score = recognize_face_knn("test_images/Andy Samberg_91.jpg", k=3, threshold=0.5)
print(name, score)
